# Modelación para la detección de cáncer de mama mediante Inteligencia Artificial y la Teoría Bayesiana

**Autores:** Rodriguez Felipe, Vargas Cristopher, Peña Angie
**Institución:** Universidad Distrital Francisco José de Caldas, Bogotá, Colombia

## ABSTRACT
This article develops the design and implementation of a modeling for the detection of breast cancer carried out by means of Artificial Intelligence and specifically Deep Learning as a starting point for its detection through taking a mammogram and in turn providing essential information for the formulation of a mathematical model for efficient and effective application, based on the Bayesian Theory for its early detection.

---

## INTRODUCCIÓN
El cáncer de mama es ahora a nivel mundial el cáncer más frecuente y la principal causa de muerte de las mujeres. Este proyecto emprende un riguroso estudio que busca lograr la detección temprana apoyados en Inteligencia Artificial y Deep Learning.

## METODOLOGÍA
Se hace uso de la metodología **SCRUM** (Pre-juego, In-juego, Pos-juego) para el desarrollo ágil de la arquitectura, garantizando iteraciones con retorno de inversión (ROI) y mitigación de riesgos. A nivel investigativo, se emplean los métodos analítico, descriptivo y sintético para comprender la esencia del modelo desde un enfoque de sistemas.

## DISEÑO Y DATASET
El Dataset usado es tomado de Kaggle/UCI, proveyendo 32 características para predecir anomalías en células mamarias (Radio, Textura, Perímetro, etc.). Cuenta con 569 muestras (357 benignas y 212 malignas).

### Fundamentación de Modelos
*   **Modelo A (SVM):** Clasificación mediante Support Vector Machine y búsqueda del hiperplano de margen máximo.
*   **Modelo B:** Regresión Lineal, Árboles de Decisión y Random Forest. La función de costo para regresión se minimiza mediante:
$$ J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)})^2 $$
*   **Modelo C (Red Neuronal Artificial - ANN):** Arquitectura multicapa con activación no lineal.
*   **Modelo D & E:** K-Nearest Neighbors (KNN), Gaussian Naive Bayes, AdaBoost y XGBoost Classifier.

## DESARROLLO E IMPLEMENTACIÓN
A continuación, realizamos la importación de librerías, carga de datos y análisis exploratorio (EDA).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 1. Carga del Dataset (Usamos el nativo de sklearn para evitar dependencias locales del CSV)
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['diagnosis'] = data.target # 0: maligno, 1: benigno

print("Primeros registros del dataset:")
display(df.head())

print("\nInformación de las columnas y nulos:")
df.info()

In [ ]:
# Gráficos Exploratorios
plt.figure(figsize=(6, 4))
sns.countplot(x='diagnosis', data=df, palette='viridis')
plt.title('Conteo de Muestras Malignas (0) y Benignas (1)')
plt.show()

plt.figure(figsize=(18, 14))
sns.heatmap(df.corr(), annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Mapa de Calor - Correlación de características')
plt.show()

### Preprocesamiento y División de Datos
Se escalan los datos usando `StandardScaler` para que los algoritmos calculen correctamente las distancias euclidianas y los pesos en las redes neuronales.

In [ ]:
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

print("Datos divididos y escalados con éxito.")
print("X_train shape:", X_train_sc.shape)
print("X_test shape:", X_test_sc.shape)

### Modelo A: Máquinas de Soporte Vectorial (SVM)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import pickle

svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_sc, y_train)
y_pred_svm = svm_model.predict(X_test_sc)

print("Precisión SVM:", accuracy_score(y_test, y_pred_svm))
print("Matriz de Confusión:\n", confusion_matrix(y_test, y_pred_svm))

# Exportación del modelo
with open('modelo_svm.pkl', 'wb') as f:
    pickle.dump(svm_model, f)
print("Modelo exportado a modelo_svm.pkl")

### Modelo B: Árboles de Decisión y Random Forest

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_sc, y_train)
print("Precisión Regresión Logística:", lr_model.score(X_test_sc, y_test))

dt_model = DecisionTreeClassifier(criterion='entropy', random_state=42)
dt_model.fit(X_train_sc, y_train)
print("Precisión Árbol de Decisión:", dt_model.score(X_test_sc, y_test))

rf_model = RandomForestClassifier(n_estimators=100, criterion='entropy', random_state=42)
rf_model.fit(X_train_sc, y_train)
print("Precisión Bosque Aleatorio:", rf_model.score(X_test_sc, y_test))

### Modelo C: Red Neuronal Artificial (ANN)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Arquitectura de la Red Neuronal
ann = Sequential()
ann.add(Dense(units=16, activation='relu', input_dim=30))
ann.add(Dense(units=16, activation='relu'))
ann.add(Dense(units=1, activation='sigmoid'))

ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entrenamiento
history = ann.fit(X_train_sc, y_train, batch_size=32, epochs=50, validation_split=0.2, verbose=0)

# Predicción
y_pred_ann = (ann.predict(X_test_sc) > 0.5)

print("\nPrecisión ANN:", accuracy_score(y_test, y_pred_ann))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Precisión por Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida por Epoch')
plt.legend()
plt.show()

### Modelo D & E: Ensamblaje y Algoritmos Probabilísticos

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_sc, y_train)
print("Precisión KNN:", knn_model.score(X_test_sc, y_test))

gnb_model = GaussianNB()
gnb_model.fit(X_train_sc, y_train)
print("Precisión Naive Bayes:", gnb_model.score(X_test_sc, y_test))

ada_model = AdaBoostClassifier(random_state=42)
ada_model.fit(X_train_sc, y_train)
print("Precisión AdaBoost:", ada_model.score(X_test_sc, y_test))

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train_sc, y_train)
print("Precisión XGBoost:", xgb_model.score(X_test_sc, y_test))

## MODELO DE TEORÍA BAYESIANA
El Teorema de Bayes calcula la probabilidad condicional de un evento dado:
$$ P(H|E) = \frac{P(E|H)P(H)}{P(E)} $$
Este teorema permite estructurar matrices de riesgo, donde cada iteración retroalimenta al sistema con estados de la naturaleza probabilísticos.

---

## CONCLUSIONES
Se logró evidenciar que el uso de determinados algoritmos de Machine Learning (como XGBoost y ANN) permite tener una alta capacidad al predecir la existencia del cáncer de mama.
Además, el planteamiento de un modelo basado en la Teoría Bayesiana brinda un enfoque sistémico que puede evolucionar al ser combinado con las matrices de confusión generadas por estos modelos.